# nb16 — RNA-Protein Deviation Characterization

Using per-protein Pearson r from masked_l1_1e4 (nb15) as RNA-protein coupling score.
Classify proteins into RNA-coupled vs RNA-independent, then characterize deviations
by tissue and cross-reference with drug response signal.

Inputs:
- `results/rna_to_protein/armB2_per_protein.csv` — per-protein Pearson r from nb15
- `data/processed/nb14_protein_ic50_correlations.csv` — protein-IC50 correlations from nb14
- `data/GDSC2/proteomics.csv` — raw protein matrix
- `data/GDSC2/gene_expression.csv` — raw RNA matrix
- `data/splits/splits.json` — LCO splits

Outputs:
- `results/deviation/protein_classification.csv` — coupled vs independent per protein
- `results/deviation/tissue_coupling_profile.csv` — median coupling score per tissue
- `results/deviation/high_deviation_drug_relevant.csv` — RNA-independent + IC50-correlated proteins

## Environment setup

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/multiomics_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

## Imports and config

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

In [ ]:
DATA_DIR      = BASE_PATH / 'data' / 'GDSC2'
PROCESSED_DIR = BASE_PATH / 'data' / 'processed'
SPLITS_DIR    = BASE_PATH / 'data' / 'splits'
NB15_DIR      = BASE_PATH / 'results' / 'rna_to_protein'
RESULTS_DIR   = BASE_PATH / 'results' / 'deviation'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

COL_CELLOSAURUS = 'cellosaurus_id'
COL_TISSUE      = 'tissue'

# Coupling thresholds
COUPLED_R_THRESHOLD     = 0.3   # protein well-predicted by RNA
INDEPENDENT_R_THRESHOLD = 0.1   # protein poorly predicted by RNA

# IC50 correlation threshold for drug relevance
IC50_CORR_THRESHOLD = 0.2

## Load data

In [ ]:
# Per-protein coupling scores from nb15 (masked_l1_1e4)
coupling_df = pd.read_csv(NB15_DIR / 'armB2_per_protein.csv')
print(f'Proteins from nb15: {len(coupling_df):,}')
print(coupling_df.head(3))

In [ ]:
# Raw omics matrices
rna     = pd.read_csv(DATA_DIR / 'gene_expression.csv',  index_col=0)
protein = pd.read_csv(DATA_DIR / 'proteomics.csv',        index_col=0)

rna     = rna[~rna.index.duplicated(keep='first')].iloc[:, 1:]
protein = protein[~protein.index.duplicated(keep='first')]

print(f'RNA:     {rna.shape}')
print(f'Protein: {protein.shape}')

In [ ]:
# Response pairs (for tissue labels)
pairs = pd.read_parquet(PROCESSED_DIR / 'response_pairs.parquet')
cell_tissue = (
    pairs[[COL_CELLOSAURUS, COL_TISSUE]]
    .drop_duplicates(COL_CELLOSAURUS)
    .set_index(COL_CELLOSAURUS)
)
print(f'Cell lines with tissue labels: {len(cell_tissue):,}')
print(f'Tissues: {cell_tissue[COL_TISSUE].nunique()}')

In [ ]:
# LCO splits — use fold 0 train cell lines for deviation analysis
# (same cell lines the nb15 model was trained on — no leakage)
with open(SPLITS_DIR / 'splits.json') as f:
    folds = json.load(f)

fold         = folds[0]
train_pairs  = pairs.loc[fold['train']]
test_pairs   = pairs.loc[fold['lco_test']]
train_cells  = train_pairs[COL_CELLOSAURUS].unique().tolist()
test_cells   = test_pairs[COL_CELLOSAURUS].unique().tolist()

print(f'Train cell lines: {len(train_cells):,}')
print(f'Test  cell lines: {len(test_cells):,}')

## Step 1 — Classify proteins by RNA-coupling

Use per-protein Pearson r from nb15 masked_l1_1e4 as the coupling score.
Three classes:
- **coupled**: r ≥ 0.3 — RNA explains protein well
- **intermediate**: 0.1 ≤ r < 0.3
- **independent**: r < 0.1 — protein is largely independent of RNA

In [ ]:
def classify_coupling(r: float) -> str:
    if pd.isna(r):
        return 'independent'  # NaN = zero variance in test = no coupling signal
    if r >= COUPLED_R_THRESHOLD:
        return 'coupled'
    if r >= INDEPENDENT_R_THRESHOLD:
        return 'intermediate'
    return 'independent'


coupling_df['coupling_class'] = coupling_df['pearson_r'].apply(classify_coupling)

counts = coupling_df['coupling_class'].value_counts()
print('Protein coupling classification:')
for cls, n in counts.items():
    print(f'  {cls:15s}: {n:,} ({n/len(coupling_df)*100:.1f}%)')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
r_vals = coupling_df['pearson_r'].dropna()
ax.hist(r_vals, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
ax.axvline(COUPLED_R_THRESHOLD,     color='tomato',   linewidth=1.5, linestyle='--',
           label=f'coupled threshold (r={COUPLED_R_THRESHOLD})')
ax.axvline(INDEPENDENT_R_THRESHOLD, color='darkorange', linewidth=1.5, linestyle='--',
           label=f'independent threshold (r={INDEPENDENT_R_THRESHOLD})')
ax.set_xlabel('Pearson r (RNA → protein, masked_l1_1e4)')
ax.set_ylabel('# proteins')
ax.set_title('RNA-protein coupling score distribution')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'coupling_score_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 2 — Per-gene linear deviation

For each protein in the RNA-independent class: fit `protein ~ RNA_same_gene` across
train cell lines. Residual std = deviation magnitude. R² = how much RNA explains.
This gives a gene-level predictability score independent of the DL model.

In [ ]:
# Build aligned RNA and protein matrices on train cell lines
common_train_cells = [
    c for c in train_cells
    if c in rna.index and c in protein.index
]
print(f'Train cell lines with both RNA and protein: {len(common_train_cells):,}')

rna_tr     = rna.loc[common_train_cells]
protein_tr = protein.loc[common_train_cells]

In [ ]:
def gene_symbol(protein_col: str) -> str:
    """Extract gene symbol from ProCan column name (e.g. 'Q9BVC5;ASHWN_HUMAN' -> 'ASHWN')."""
    if ';' in protein_col:
        part = protein_col.split(';')[1]
        return part.split('_')[0]
    return protein_col


def compute_linear_deviation(protein_col: str,
                              rna_df: pd.DataFrame,
                              protein_df: pd.DataFrame) -> dict:
    """Fit protein ~ RNA_same_gene across cell lines.
    Returns slope, R², residual std, n_observed.
    """
    gene = gene_symbol(protein_col)

    # Match gene symbol to RNA column
    rna_cols = [c for c in rna_df.columns if c.startswith(gene + ' ') or c == gene]
    if not rna_cols:
        return {'protein': protein_col, 'gene': gene, 'rna_col': None,
                'slope': np.nan, 'r2': np.nan, 'residual_std': np.nan, 'n_observed': 0}

    rna_col   = rna_cols[0]
    y         = protein_df[protein_col].values.astype(float)
    x         = rna_df[rna_col].values.astype(float)

    # Keep only rows where both are observed
    valid = ~(np.isnan(x) | np.isnan(y))
    if valid.sum() < 10:
        return {'protein': protein_col, 'gene': gene, 'rna_col': rna_col,
                'slope': np.nan, 'r2': np.nan, 'residual_std': np.nan,
                'n_observed': valid.sum()}

    x_fit, y_fit = x[valid].reshape(-1, 1), y[valid]
    model        = LinearRegression().fit(x_fit, y_fit)
    y_pred       = model.predict(x_fit)
    residuals    = y_fit - y_pred

    return {
        'protein':      protein_col,
        'gene':         gene,
        'rna_col':      rna_col,
        'slope':        model.coef_[0],
        'r2':           r2_score(y_fit, y_pred),
        'residual_std': residuals.std(),
        'n_observed':   valid.sum(),
    }

In [ ]:
# Run on all proteins (not just independent — gives full picture for comparison)
# Subset protein columns to those that exist in protein_tr
protein_cols_available = [
    p for p in coupling_df['protein'].tolist()
    if p in protein_tr.columns
]
print(f'Proteins to compute linear deviation for: {len(protein_cols_available):,}')

deviation_rows = []
for i, pcol in enumerate(protein_cols_available):
    deviation_rows.append(compute_linear_deviation(pcol, rna_tr, protein_tr))
    if (i + 1) % 500 == 0:
        print(f'  {i+1:,} / {len(protein_cols_available):,}')

deviation_df = pd.DataFrame(deviation_rows)
print(f'Done. Shape: {deviation_df.shape}')

In [ ]:
# Merge coupling scores with linear deviation stats
protein_df_full = coupling_df.merge(deviation_df, on='protein', how='left')
protein_df_full.to_csv(RESULTS_DIR / 'protein_classification.csv', index=False)

print(f'Proteins with linear deviation computed: {deviation_df["r2"].notna().sum():,}')
print(f'Proteins without RNA match: {deviation_df["r2"].isna().sum():,}')
print()
print('Linear R² summary (protein ~ RNA_same_gene, train cell lines):')
print(protein_df_full['r2'].describe().round(3))

In [ ]:
# R² distribution by coupling class
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
classes = ['coupled', 'intermediate', 'independent']
colors  = ['steelblue', 'darkorange', 'tomato']

for ax, cls, col in zip(axes, classes, colors):
    subset = protein_df_full[protein_df_full['coupling_class'] == cls]['r2'].dropna()
    ax.hist(subset.clip(-1, 1), bins=50, color=col, edgecolor='none', alpha=0.8)
    ax.set_title(f'{cls} (n={len(subset):,})')
    ax.set_xlabel('Linear R² (protein ~ RNA)')
    ax.set_ylabel('# proteins')
    ax.axvline(subset.median(), color='black', linewidth=1.2, linestyle='--',
               label=f'median={subset.median():.3f}')
    ax.legend(fontsize=8)

plt.suptitle('Linear R² by coupling class (train cell lines)', y=1.02)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'r2_by_coupling_class.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 3 — Tissue stratification

For each tissue type: compute the median coupling score across all proteins.
Identifies which cancer types have the most post-transcriptional dysregulation.

In [ ]:
# Build per-cell-line coupling profile:
# for each cell line, compute mean RNA-protein Pearson r across all proteins
# using the raw protein and RNA matrices directly

common_cells = [
    c for c in rna.index
    if c in protein.index and c in cell_tissue.index
]
print(f'Cell lines with RNA + protein + tissue: {len(common_cells):,}')

# Find proteins that have a matched RNA gene
matched_proteins = protein_df_full[
    protein_df_full['rna_col'].notna()
][['protein', 'rna_col', 'coupling_class', 'pearson_r']].copy()
print(f'Proteins with matched RNA gene: {len(matched_proteins):,}')

In [ ]:
# Per-cell-line: compute mean RNA-protein correlation across matched genes
# Use a sample of proteins for speed (top 1000 by n_observed)
sample_proteins = matched_proteins.nlargest(1000, 'pearson_r').head(500)

cell_coupling_scores = []
rna_sub     = rna.loc[common_cells]
protein_sub = protein.loc[common_cells]

for cell in common_cells:
    rs = []
    for _, row in sample_proteins.iterrows():
        pval = protein_sub.loc[cell, row['protein']] if row['protein'] in protein_sub.columns else np.nan
        rval = rna_sub.loc[cell, row['rna_col']]     if row['rna_col']  in rna_sub.columns     else np.nan
        if not (np.isnan(pval) or np.isnan(rval)):
            rs.append((pval, rval))
    if len(rs) > 10:
        pv, rv = zip(*rs)
        r, _   = pearsonr(pv, rv)
        cell_coupling_scores.append({'cellosaurus_id': cell, 'mean_coupling_r': r})

cell_coupling_df = pd.DataFrame(cell_coupling_scores).set_index('cellosaurus_id')
cell_coupling_df = cell_coupling_df.join(cell_tissue)
print(f'Cell lines with coupling score: {len(cell_coupling_df):,}')

In [ ]:
# Aggregate by tissue
tissue_coupling = (
    cell_coupling_df
    .groupby(COL_TISSUE)['mean_coupling_r']
    .agg(['median', 'mean', 'count'])
    .rename(columns={'median': 'median_coupling_r', 'mean': 'mean_coupling_r', 'count': 'n_cell_lines'})
    .query('n_cell_lines >= 5')  # at least 5 cell lines per tissue
    .sort_values('median_coupling_r')
)
tissue_coupling.to_csv(RESULTS_DIR / 'tissue_coupling_profile.csv')

print('Tissues with LOWEST RNA-protein coupling (most post-transcriptional dysregulation):')
display(tissue_coupling.head(10).round(3))
print()
print('Tissues with HIGHEST RNA-protein coupling:')
display(tissue_coupling.tail(10).round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
tissues = tissue_coupling.index.tolist()
vals    = tissue_coupling['median_coupling_r'].values
colors  = ['tomato' if v < vals.mean() else 'steelblue' for v in vals]

ax.barh(tissues, vals, color=colors, edgecolor='none', alpha=0.8)
ax.axvline(vals.mean(), color='black', linewidth=1, linestyle='--',
           label=f'mean={vals.mean():.3f}')
ax.set_xlabel('Median RNA-protein coupling r')
ax.set_title('RNA-protein coupling by tissue type')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tissue_coupling_profile.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 4 — Cross-reference with IC50 correlations

Load protein-IC50 correlations from nb14.
Find proteins where: (a) RNA-independent (r < 0.1) AND (b) protein-IC50 |r| > threshold.
These are the biologically most interesting candidates:
post-transcriptionally regulated AND drug-relevant.

In [ ]:
ic50_corr = pd.read_csv(PROCESSED_DIR / 'nb14_protein_ic50_correlations.csv')
print(f'Protein-IC50 correlations loaded: {ic50_corr.shape}')
print(ic50_corr.head(3))

In [ ]:
# Merge coupling classification with IC50 correlation
# ic50_corr expected columns: protein, ic50_pearson_r (or similar)
# Adapt column name if different
ic50_col = [c for c in ic50_corr.columns if 'corr' in c.lower() or 'pearson' in c.lower() or 'r' == c.lower()]
print(f'IC50 correlation column detected: {ic50_col}')

merged = protein_df_full.merge(
    ic50_corr.rename(columns={ic50_col[0]: 'ic50_corr'})[['protein', 'ic50_corr']],
    on='protein', how='inner'
)
print(f'Proteins with both coupling score and IC50 correlation: {len(merged):,}')

In [ ]:
# Identify high-deviation, drug-relevant proteins
high_dev_drug_relevant = merged[
    (merged['coupling_class'] == 'independent') &
    (merged['ic50_corr'].abs() >= IC50_CORR_THRESHOLD)
].copy()

high_dev_drug_relevant = high_dev_drug_relevant.sort_values('ic50_corr', key=abs, ascending=False)
high_dev_drug_relevant.to_csv(RESULTS_DIR / 'high_deviation_drug_relevant.csv', index=False)

print(f'RNA-independent + drug-relevant proteins: {len(high_dev_drug_relevant):,}')
print()
print('Top 20 by |IC50 correlation|:')
display(high_dev_drug_relevant[
    ['protein', 'gene', 'pearson_r', 'r2', 'residual_std', 'ic50_corr']
].head(20).round(3))

In [ ]:
# Scatter: coupling score vs IC50 correlation — shows the quadrant of interest
fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(
    merged['pearson_r'], merged['ic50_corr'].abs(),
    alpha=0.2, s=6, color='steelblue', rasterized=True
)

# Highlight the interesting quadrant
ax.axvline(INDEPENDENT_R_THRESHOLD, color='tomato',   linewidth=1.2, linestyle='--',
           label=f'independent threshold (r={INDEPENDENT_R_THRESHOLD})')
ax.axhline(IC50_CORR_THRESHOLD,     color='darkorange', linewidth=1.2, linestyle='--',
           label=f'IC50 threshold (|r|={IC50_CORR_THRESHOLD})')

# Annotate top candidates
for _, row in high_dev_drug_relevant.head(10).iterrows():
    ax.annotate(
        row['gene'],
        xy=(row['pearson_r'], abs(row['ic50_corr'])),
        fontsize=7, alpha=0.8,
        xytext=(4, 4), textcoords='offset points'
    )

ax.set_xlabel('RNA-protein coupling (Pearson r, masked_l1_1e4)')
ax.set_ylabel('|Protein-IC50 correlation|')
ax.set_title('RNA-independent proteins with drug response signal\n'
             'Bottom-left quadrant = post-transcriptionally regulated + drug-relevant')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'coupling_vs_ic50_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

In [ ]:
print('=' * 60)
print('nb16 SUMMARY')
print('=' * 60)
print(f'Total proteins analysed:         {len(protein_df_full):,}')
print()
for cls in ['coupled', 'intermediate', 'independent']:
    n = (protein_df_full['coupling_class'] == cls).sum()
    print(f'  {cls:15s}: {n:,} ({n/len(protein_df_full)*100:.1f}%)')
print()
print(f'Tissues profiled:                {len(tissue_coupling):,}')
lowest  = tissue_coupling.index[0]
highest = tissue_coupling.index[-1]
print(f'Lowest  coupling tissue:         {lowest} '
      f'(r={tissue_coupling.loc[lowest, "median_coupling_r"]:.3f})')
print(f'Highest coupling tissue:         {highest} '
      f'(r={tissue_coupling.loc[highest, "median_coupling_r"]:.3f})')
print()
print(f'RNA-independent + drug-relevant: {len(high_dev_drug_relevant):,} proteins')
print()
print('Outputs saved to:', RESULTS_DIR)
print('  protein_classification.csv')
print('  tissue_coupling_profile.csv')
print('  high_deviation_drug_relevant.csv')